# 02 — Popularity Recommender

Implements the **IMDb Bayesian Weighted Rating** formula:

```
score = (v / (v + m)) * R  +  (m / (v + m)) * C
```

where:
- `R` = movie's average rating
- `v` = movie's vote count
- `C` = global mean rating across all movies with at least one rating
- `m` = 90th-percentile vote count (minimum-votes threshold)

**Optimisations applied**
- Filter `num_ratings >= m` *before* computing the Bayesian score — avoids wasted
  computation on ~92 % of movies that will be discarded anyway.
- `get_top_movies()` no longer redundantly re-sorts the already-sorted frame.
- `sort=False` on `groupby` skips unnecessary index sorting during aggregation.
- Dtype hints on CSV read (`int32` / `float32`) cut ratings memory ~50 %.

In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
movies = pd.read_csv(
    "../data/movies.csv",
    dtype={"movieId": "int32"},
    encoding="utf-8",
)

ratings = pd.read_csv(
    "../data/ratings.csv",
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32"},
    usecols=["userId", "movieId", "rating"],
    encoding="utf-8",
)

In [ ]:
# Aggregate rating stats once — reused for C, m, and filtering.
movie_stats = (
    ratings
    .groupby("movieId", sort=False)
    .agg(avg_rating=("rating", "mean"), num_ratings=("rating", "count"))
    .reset_index()
)

movie_stats.head()

In [ ]:
# C = mean rating across all movies with at least one rating.
C = float(movie_stats["avg_rating"].mean())

# m = 90th-percentile vote count (minimum-votes threshold).
m = float(movie_stats["num_ratings"].quantile(0.90))

print(f"C (global mean rating) = {C:.4f}")
print(f"m (90th-pct vote count) = {m:.0f}")

In [ ]:
# ── Filter BEFORE scoring ─────────────────────────────────────────────────────
# Only movies with enough votes are eligible. Filtering first avoids computing
# the Bayesian score for the ~92 % of movies that would be dropped anyway.
qualified = movie_stats[movie_stats["num_ratings"] >= m].copy()
print(f"Movies qualifying for recommendation: {len(qualified):,}")

v = qualified["num_ratings"]
qualified["score"] = (v / (v + m)) * qualified["avg_rating"] + (m / (v + m)) * C

In [ ]:
# Merge movie metadata and sort by score (descending).
qualified = qualified.merge(movies, on="movieId")
qualified = qualified.sort_values("score", ascending=False).reset_index(drop=True)

qualified[["title", "avg_rating", "num_ratings", "score"]].head(20)

In [ ]:
def get_top_movies(n: int = 10) -> pd.DataFrame:
    """Return the top-n movies by Bayesian weighted popularity score.

    `qualified` is already sorted — no redundant sort_values() needed.
    """
    return (
        qualified[["title", "avg_rating", "num_ratings", "score"]]
        .head(n)
        .reset_index(drop=True)
    )

In [ ]:
get_top_movies(10)

In [ ]:
os.makedirs("../outputs", exist_ok=True)
qualified.to_csv("../outputs/top_movies.csv", index=False)
print("Saved → ../outputs/top_movies.csv")